In [21]:
spark.stop()

In [1]:
from utils import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = instantiate_spark_sedona("10g")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 10:17:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/03 10:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/03 10:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark Session and SedonaContext have been successfully initiated.


In [2]:
quadkey_buil_floor_area = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Gee_Buildings/quadkey_buildings_metric.parquet")
quadkey_geom_names = spark.read.csv("quadkeys_w_admin_names.csv",header=True)
quadkey_buil_floor_area = quadkey_buil_floor_area.join(
    quadkey_geom_names,
    on = 'quadkey',
    how = 'inner'
)
quadkey_buil_floor_area.count()

981705

In [3]:
quadkey_buil_floor_area.printSchema()

root
 |-- quadkey: string (nullable = true)
 |-- sum_floor_area_sqm: double (nullable = true)
 |-- adm3_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- adm1_name: string (nullable = true)



In [10]:
pop_quadtile = spark.read.parquet('abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Population_filtered/thailand_population_quadkey_indexed.parquet')
pop_quadtile = pop_quadtile.groupby(col('qk16').alias('quadkey')).agg(sum(col('tha_general_2020')).alias("Quadkey_Pop"))
pop_quadtile.show()

[Stage 29:====================>                                  (12 + 20) / 32]

+----------------+------------------+
|         quadkey|       Quadkey_Pop|
+----------------+------------------+
|1322120122120110|         32.218992|
|1322120033120001|          2.242451|
|1322031122121103| 151.1293869999998|
|1322120033031013|24.215984999999996|
|1322031133021012|14.585018000000002|
|1322120132120013| 37.33065000000001|
|1322120032130102|           2.70122|
|1322120122120030|23.269271999999994|
|1322121122030123|11.764437999999998|
|1322031032131211|           4.33782|
|1322031133131311| 97.75779199999994|
|1322031123031200|29.072247999999988|
|1322120023130312|          5.253534|
|1322121022130220|16.982575999999998|
|1322121023020230|15.074513999999999|
|1322120032130230|21.609759999999998|
|1322121032131331|          6.209976|
|1322120133131331|          4.655402|
|1322120033030223| 94.17327499999998|
|1322120033121323|         21.102021|
+----------------+------------------+
only showing top 20 rows



In [12]:
pop_quadtile.filter(col('quadkey') == '1322033101303300').show()

[Stage 32:======================================================> (31 + 1) / 32]

+----------------+-----------+
|         quadkey|Quadkey_Pop|
+----------------+-----------+
|1322033101303300|  19.081168|
+----------------+-----------+



In [5]:
pop_buil_quadkey = quadkey_buil_floor_area.join(
    pop_quadtile,
    on = 'quadkey',
    how = 'left'
)
pop_buil_quadkey.count()

981705

In [6]:
pop_buil_quadkey.show(5)

+----------------+------------------+---------+----------+------------+-----------+
|         quadkey|sum_floor_area_sqm|adm3_name| adm2_name|   adm1_name|Quadkey_Pop|
+----------------+------------------+---------+----------+------------+-----------+
|1322010330330112|120.31849027728747|Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL|
|1322010331201320|17.154244963232237|Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL|
|1322010331201323|192.72464318952922|Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL|
|1322010331202200| 1561.648755833432|Na Pu Pom|Pang Mapha|Mae Hong Son|   7.938592|
|1322010331222012| 35.54481692685274|Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL|
+----------------+------------------+---------+----------+------------+-----------+
only showing top 5 rows



In [8]:
null_quads = pop_buil_quadkey.filter(col('Quadkey_Pop').isNull())
non_null_quads = pop_buil_quadkey.filter(col('Quadkey_Pop').isNotNull())
null_quads.count(), non_null_quads.count()

(224945, 756760)

In [9]:
null_quads = null_quads.withColumn("ulimit", col('sum_floor_area_sqm')*1.2) \
                        .withColumn("llimit", col('sum_floor_area_sqm')*0.8)
null_quads.show(5)

[Stage 60:====================>                                  (12 + 20) / 32]

+----------------+------------------+----------+----------+------------+-----------+------------------+------------------+
|         quadkey|sum_floor_area_sqm| adm3_name| adm2_name|   adm1_name|Quadkey_Pop|            ulimit|            llimit|
+----------------+------------------+----------+----------+------------+-----------+------------------+------------------+
|1322010330330112|120.31849027728747| Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL|144.38218833274496| 96.25479222182997|
|1322010331201320|17.154244963232237| Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL|20.585093955878683|13.723395970585791|
|1322010331201323|192.72464318952922| Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL|231.26957182743504| 154.1797145516234|
|1322010331222012| 35.54481692685274| Na Pu Pom|Pang Mapha|Mae Hong Son|       NULL| 42.65378031222328|28.435853541482192|
|1322010331232100|237.50708881119294|Pang Mapha|Pang Mapha|Mae Hong Son|       NULL|285.00850657343153|190.00567104895435|
+---------------

In [13]:
nq = null_quads.alias("nq")
nnq = non_null_quads.alias("nnq")

join = nq.join(
    nnq,
    (
        (col("nq.adm1_name") == col("nnq.adm1_name")) &
        (col("nq.adm2_name") == col("nnq.adm2_name")) &
        (col("nq.adm3_name") == col("nnq.adm3_name")) &
        (col("nq.ulimit") >= col("nnq.sum_floor_area_sqm")) &
        (col("nq.llimit") <= col("nnq.sum_floor_area_sqm"))
    ),
    "left"
).select(
    col("nq.*"),
    col("nnq.sum_floor_area_sqm").alias("neighbour_sum_floor_area_sqm"),
    col("nnq.Quadkey_Pop").alias("neighbour_quad_pop")
)


# join.write.mode('overwrite').parquet('abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Population_filtered/intermediate_state_data')


In [7]:
join = spark.read.parquet('abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Population_filtered/intermediate_state_data')
join.printSchema()
# join.filter(col('quadkey') == '1322033101303300').show()

root
 |-- quadkey: string (nullable = true)
 |-- sum_floor_area_sqm: double (nullable = true)
 |-- adm3_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- Quadkey_Pop: double (nullable = true)
 |-- ulimit: double (nullable = true)
 |-- llimit: double (nullable = true)
 |-- neighbour_sum_floor_area_sqm: double (nullable = true)
 |-- neighbour_quad_pop: double (nullable = true)



In [29]:
join = spark.read.parquet('abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Population_filtered/intermediate_state_data')
join_joined1 = join.filter(col('neighbour_sum_floor_area_sqm').isNotNull())
join_joined1.select(col('quadkey')).distinct().count()

192271

In [33]:
join_joined1_grouped = join_joined1.groupby(col('quadkey'), col('sum_floor_area_sqm'), col('adm3_name'), col('adm2_name'), col('adm1_name')).agg(
    avg(col('neighbour_quad_pop')).alias('Quadkey_Pop')
)
join_joined1_grouped.show()

+----------------+------------------+------------+-------------+----------------+------------------+
|         quadkey|sum_floor_area_sqm|   adm3_name|    adm2_name|       adm1_name|       Quadkey_Pop|
+----------------+------------------+------------+-------------+----------------+------------------+
|1322102130222203|184.35018146434737|    Non Sila|     Pak Khat|       Bueng Kan| 6.351815076923078|
|1322012131331303|173.13360288403743|  Kong Khaek|    Mae Chaem|      Chiang Mai| 2.600553466666667|
|1322011322102203|243.07608951943956|     Mae Loi|       Thoeng|      Chiang Rai| 4.217696363636363|
|1322011322103022| 643.6055720756533|     Mae Loi|       Thoeng|      Chiang Rai| 9.700701636363634|
|1322102332031303| 705.1520181474128|      Samran|     Sam Chai|         Kalasin|14.035160500000002|
|1322102332013103|176.31463026067928|      Samran|     Sam Chai|         Kalasin|11.112166333333333|
|1322031011003001|376.84264124239263|Phran Kratai| Phran Kratai|  Kamphaeng Phet|        8.

In [38]:
join_unjoined = join.filter(col('neighbour_sum_floor_area_sqm').isNull()).drop(col('neighbour_sum_floor_area_sqm'), col('neighbour_quad_pop'))
join_unjoined.count()

32674

In [39]:
join_unjoined.printSchema()

root
 |-- quadkey: string (nullable = true)
 |-- sum_floor_area_sqm: double (nullable = true)
 |-- adm3_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- Quadkey_Pop: double (nullable = true)
 |-- ulimit: double (nullable = true)
 |-- llimit: double (nullable = true)



In [42]:
nq = join_unjoined.alias("nq")
nnq = non_null_quads.alias("nnq")

join2 = nq.join(
    nnq,
    (
        (col("nq.adm1_name") == col("nnq.adm1_name")) &
        (col("nq.adm2_name") == col("nnq.adm2_name")) &
        (col("nq.ulimit") >= col("nnq.sum_floor_area_sqm")) &
        (col("nq.llimit") <= col("nnq.sum_floor_area_sqm"))
    ),
    "left"
).select(
    col("nq.*"),
    col("nnq.sum_floor_area_sqm").alias("neighbour_sum_floor_area_sqm"),
    col("nnq.Quadkey_Pop").alias("neighbour_quad_pop")
)

join2.persist()


DataFrame[quadkey: string, sum_floor_area_sqm: double, adm3_name: string, adm2_name: string, adm1_name: string, Quadkey_Pop: double, ulimit: double, llimit: double, neighbour_sum_floor_area_sqm: double, neighbour_quad_pop: double]

In [44]:
join_joined2 = join2.filter(col('neighbour_sum_floor_area_sqm').isNotNull())
join_joined2.select(col('quadkey')).distinct().count()

29817

In [48]:
join_joined2_grouped = join_joined2.groupby(col('quadkey'), col('sum_floor_area_sqm'), col('adm3_name'), col('adm2_name'), col('adm1_name')).agg(
    avg(col('neighbour_quad_pop')).alias('Quadkey_Pop')
)
join_joined2_grouped.show()

+----------------+------------------+------------+---------+----------+------------------+
|         quadkey|sum_floor_area_sqm|   adm3_name|adm2_name| adm1_name|       Quadkey_Pop|
+----------------+------------------+------------+---------+----------+------------------+
|1322011322031120| 23.34818168412724|     Pa Daet|  Pa Daet|Chiang Rai|          2.127667|
|1322011322211221|150.30673792666914|Si Pho Ngoen|  Pa Daet|Chiang Rai| 3.864411391304349|
|1322011322210233| 39.92923282817152|Si Pho Ngoen|  Pa Daet|Chiang Rai|         3.1915005|
|1322011322210210|209.30302077098526|Si Pho Ngoen|  Pa Daet|Chiang Rai|  4.61987942857143|
|1322011322213010|122.80150959525525|Si Pho Ngoen|  Pa Daet|Chiang Rai| 3.590438062500001|
|1322011322122232|34.688331195879265|   San Makha|  Pa Daet|Chiang Rai|          2.127667|
|1322011322300131|26.032292448025252|   San Makha|  Pa Daet|Chiang Rai|          2.127667|
|1322011322213011| 35.97660161958373|   San Makha|  Pa Daet|Chiang Rai|          2.127667|

In [49]:
join_joined2_grouped.count()

29817

In [50]:
join2_unjoined = join2.filter(col('neighbour_sum_floor_area_sqm').isNull()).drop(col('neighbour_sum_floor_area_sqm'), col('neighbour_quad_pop'))
join2_unjoined.count()

2857

In [51]:
nq = join2_unjoined.alias("nq")
nnq = non_null_quads.alias("nnq")

join3 = nq.join(
    nnq,
    (
        (col("nq.adm1_name") == col("nnq.adm1_name")) &
        (col("nq.ulimit") >= col("nnq.sum_floor_area_sqm")) &
        (col("nq.llimit") <= col("nnq.sum_floor_area_sqm"))
    ),
    "left"
).select(
    col("nq.*"),
    col("nnq.sum_floor_area_sqm").alias("neighbour_sum_floor_area_sqm"),
    col("nnq.Quadkey_Pop").alias("neighbour_quad_pop")
)

join3.persist()


DataFrame[quadkey: string, sum_floor_area_sqm: double, adm3_name: string, adm2_name: string, adm1_name: string, Quadkey_Pop: double, ulimit: double, llimit: double, neighbour_sum_floor_area_sqm: double, neighbour_quad_pop: double]

In [52]:
join_joined3 = join3.filter(col('neighbour_sum_floor_area_sqm').isNotNull())
join_joined3.select(col('quadkey')).distinct().count()

2834

In [53]:
join_joined3_grouped = join_joined3.groupby(col('quadkey'), col('sum_floor_area_sqm'), col('adm3_name'), col('adm2_name'), col('adm1_name')).agg(
    avg(col('neighbour_quad_pop')).alias('Quadkey_Pop')
)
join_joined3_grouped.show()

+----------------+------------------+------------+-----------+------------+------------------+
|         quadkey|sum_floor_area_sqm|   adm3_name|  adm2_name|   adm1_name|       Quadkey_Pop|
+----------------+------------------+------------+-----------+------------+------------------+
|1322033112131021| 70.68146158770529| Song Khlong|Bang Pakong|Chachoengsao|5.2506715326086955|
|1322033113020310| 73.81485912437827| Bang Pakong|Bang Pakong|Chachoengsao| 5.310061217391304|
|1322033111333202| 76.86735978411113|  Laem Pradu|    Ban Pho|Chachoengsao| 5.320456282828282|
|1322033111332313|43.251811108020405|  Laem Pradu|    Ban Pho|Chachoengsao| 4.393631259999999|
|1322033111332131| 46.30231329587456| Khlong Khut|    Ban Pho|Chachoengsao| 4.604056793650792|
|1322033111221331| 85.09924868621655|  Theppharat|    Ban Pho|Chachoengsao|5.3156128983050825|
|1322033111203221|58.610679883733354|      Ko Rai|    Ban Pho|Chachoengsao| 4.587107367088609|
|1322122000202211|15.472622340631068|  Mueang Mai|

In [54]:
join3_unjoined = join3.filter(col('neighbour_sum_floor_area_sqm').isNull()).drop(col('neighbour_sum_floor_area_sqm'), col('neighbour_quad_pop'))
join3_unjoined.count()

23

In [56]:
final_pop = non_null_quads.unionByName(join_joined1_grouped).unionByName(join_joined2_grouped).unionByName(join_joined3_grouped)
final_pop.count()

981682

In [ ]:
final_pop.select(col('quadkey'), col('Quadkey_Pop'))

In [57]:
final_pop.select(sum('Quadkey_Pop')).show()

[Stage 443:================================================>   (417 + 29) / 446]

+-------------------+
|   sum(Quadkey_Pop)|
+-------------------+
|7.099864767175704E7|
+-------------------+



In [60]:
# final_pop.write.parquet('abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Population_filtered/Quadkey_wise_population_updated')

In [61]:
spark.stop()

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.
